<a href="https://colab.research.google.com/github/24cs066-hash/spam_sms/blob/main/spam_sms_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
df = pd.read_csv('/content/sample_data/california_housing_train.csv')
print(df.head())
print(df.columns.tolist())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -114.31     34.19                15.0       5612.0          1283.0   
1    -114.47     34.40                19.0       7650.0          1901.0   
2    -114.56     33.69                17.0        720.0           174.0   
3    -114.57     33.64                14.0       1501.0           337.0   
4    -114.57     33.57                20.0       1454.0           326.0   

   population  households  median_income  median_house_value  
0      1015.0       472.0         1.4936             66900.0  
1      1129.0       463.0         1.8200             80100.0  
2       333.0       117.0         1.6509             85700.0  
3       515.0       226.0         3.1917             73400.0  
4       624.0       262.0         1.9250             65500.0  
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


In [ ]:
filename = '/content/sample_data/california_housing_train.csv'
df = pd.read_csv(filename, encoding="latin-1")

print("Dataset loaded successfully!")

print("Number of rows and columns:")
print(df.shape)

print("Column names:")
print(df.columns.tolist())

print("First five rows:")
display(df.head())

Dataset loaded successfully!
Number of rows and columns:
(17000, 9)
Column names:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']
First five rows:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


In [ ]:
# Display the first 5 messages
display(df.head())

# Display all column names
print("Column names:")
print(df.columns.tolist())

# Display the dataset size
print("Dataset size:", df.shape)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


Column names:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']
Dataset size: (17000, 9)


In [ ]:
print("Column names:", df.columns.tolist())

# Identify common column names
columns = {
    str(col).strip().lower(): col
    for col in df.columns
}

label_candidates = ["label", "v1", "category", "class", "type"]
message_candidates = ["message", "v2", "sms", "text", "message_body"]

label_col = next(
    (columns[c] for c in label_candidates if c in columns),
    None
)

message_col = next(
    (columns[c] for c in message_candidates if c in columns),
    None
)

print("Label column:", label_col)
print("Message column:", message_col)

Column names: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']
Label column: None
Message column: None


In [ ]:
if label_col is not None and message_col is not None:
    df = df[[label_col, message_col]].copy()
    # Rename the columns to simple names
    df.columns = ["label", "message"]
    display(df.head())
else:
    print("Error: Could not find matching text classification columns (label/message) in this dataset.")
    print("Current columns are:", df.columns.tolist())

Error: Could not find matching text classification columns (label/message) in this dataset.
Current columns are: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Safely perform cleaning only if 'label' and 'message' columns exist
if "label" in df.columns and "message" in df.columns:
    # Remove rows with missing labels or messages
    df = df.dropna(subset=["label", "message"])

    # Remove messages that are empty or contain only spaces
    df["message"] = df["message"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    df = df[df["message"] != ""]

    # Keep only the two valid classes
    df = df[df["label"].isin(["ham", "spam"])]

    # Remove duplicate messages
    df = df.drop_duplicates(subset=["message"])

    # Reset row numbering
    df = df.reset_index(drop=True)

    print("\nDataset size after cleaning:", df.shape)
    print("\nMissing values after cleaning:")
    print(df.isnull().sum())
else:
    print("\nSkipping text-cleaning steps: 'label' and 'message' columns are not present in this dataset.")

Missing values before cleaning:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64

Skipping text-cleaning steps: 'label' and 'message' columns are not present in this dataset.


In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Safely perform cleaning only if 'label' and 'message' columns exist
if "label" in df.columns and "message" in df.columns:
    # Remove rows with missing labels or messages
    df = df.dropna(subset=["label", "message"])

    # Remove messages that are empty or contain only spaces
    df["message"] = df["message"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    df = df[df["message"] != ""]

    # Keep only the two valid classes
    df = df[df["label"].isin(["ham", "spam"])]

    # Remove duplicate messages
    df = df.drop_duplicates(subset=["message"])

    # Reset row numbering
    df = df.reset_index(drop=True)

    print("\nDataset size after cleaning:", df.shape)
    print("\nMissing values after cleaning:")
    print(df.isnull().sum())
else:
    print("\nSkipping text-cleaning steps: 'label' and 'message' columns are not present in this dataset.")

Missing values before cleaning:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64

Skipping text-cleaning steps: 'label' and 'message' columns are not present in this dataset.


In [ ]:
# Safely map labels only if 'label' and 'message' columns are present
if "label" in df.columns and "message" in df.columns:
    df["label_id"] = df["label"].map({
        "ham": 0,
        "spam": 1
    })
    print(df[["label", "label_id", "message"]].head(10))
else:
    print("Skipping mapping: 'label' and 'message' columns are not present in this dataset.")

Skipping mapping: 'label' and 'message' columns are not present in this dataset.


In [ ]:
# Safely extract X and y only if the required columns are present
if "message" in df.columns and "label_id" in df.columns:
    X = df["message"].values

    # y contains the numerical labels (0 = ham, 1 = spam)
    y = df["label_id"].values

    print("First SMS message:")
    print(X[0])

    print("\nIts label:")
    print(y[0])

    print("\nTotal messages:", len(X))
    print("Total labels:", len(y))
else:
    print("Skipping extraction: 'message' and 'label_id' columns are not present in this dataset.")

Skipping extraction: 'message' and 'label_id' columns are not present in this dataset.


In [ ]:
from sklearn.model_selection import train_test_split

# Safely perform split only if X and y have been defined
if 'X' in globals() and 'y' in globals():
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print("Training messages:", len(X_train))
    print("Testing messages:", len(X_test))

    print("Training labels:", len(y_train))
    print("Testing labels:", len(y_test))
else:
    print("Skipping train-test split: 'X' and 'y' variables are not defined.")

Skipping train-test split: 'X' and 'y' variables are not defined.


In [ ]:
import tensorflow as tf

from tensorflow.keras.layers import TextVectorization

# Maximum vocabulary size
MAX_WORDS = 5000

# Fixed length of each SMS sequence
MAX_LENGTH = 100

# Create the text vectorizer
vectorizer = TextVectorization(
    max_tokens=MAX_WORDS,
    output_mode="int",
    output_sequence_length=MAX_LENGTH,
    standardize="lower_and_strip_punctuation"
)

print("Text vectorizer created successfully!")

Text vectorizer created successfully!


In [ ]:
# Safely adapt the vectorizer only if X_train is defined
if 'X_train' in globals():
    vectorizer.adapt(X_train)

    # Display some words in the vocabulary
    vocabulary = vectorizer.get_vocabulary()

    print("Vocabulary size:", len(vocabulary))
    print("\nFirst 30 vocabulary entries:")
    print(vocabulary[:30])
else:
    print("Skipping vectorizer adaptation: 'X_train' is not defined.")

Skipping vectorizer adaptation: 'X_train' is not defined.


In [ ]:
# Safely convert messages only if X_train and X_test are defined
if 'X_train' in globals() and 'X_test' in globals():
    X_train_seq = vectorizer(X_train).numpy()

    # Convert testing messages into numerical sequences
    X_test_seq = vectorizer(X_test).numpy()

    print("Training sequence shape:", X_train_seq.shape)
    print("Testing sequence shape:", X_test_seq.shape)

    print("\nFirst original message:")
    print(X_train[0])

    print("\nIts numerical sequence:")
    print(X_train_seq[0])
else:
    print("Skipping conversion: 'X_train' and 'X_test' variables are not defined.")

Skipping conversion: 'X_train' and 'X_test' variables are not defined.


In [ ]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    Dense,
    Dropout,
    Input
)


In [ ]:
model = Sequential([
    Input(shape=(MAX_LENGTH,)),

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=64,
        mask_zero=True
    ),

    SimpleRNN(64, return_sequences=False),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 330,369 (1.26 MB)

 Trainable params: 330,369 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

Model compiled successfully!


In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 330,369 (1.26 MB)

 Trainable params: 330,369 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Only attempt to train if training data variables have been defined
if 'X_train_seq' in globals() and 'y_train' in globals():
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_seq,
        y_train,
        epochs=15,
        batch_size=32,
        validation_split=0.20,
        callbacks=[early_stop],
        verbose=1
    )
else:
    print("Skipping training: 'X_train_seq' and/or 'y_train' are not defined.")
    print("Please upload a text classification dataset and run the preceding cells first.")

Skipping training: 'X_train_seq' and/or 'y_train' are not defined.
Please upload a text classification dataset and run the preceding cells first.


In [ ]:
# Safely predict only if test sequences are defined
if 'X_test_seq' in globals():
    y_prob = model.predict(X_test_seq)
    print("First 10 predicted probabilities:")
    print(y_prob[:10])
else:
    print("Skipping prediction: 'X_test_seq' is not defined.")
    print("Please upload a text classification dataset and run the preceding cells first.")

Skipping prediction: 'X_test_seq' is not defined.
Please upload a text classification dataset and run the preceding cells first.


In [ ]:
import numpy as np

# Safely execute evaluation steps only if y_prob and y_test are defined
if 'y_prob' in globals() and 'y_test' in globals():
    # Convert probabilities into class labels
    y_pred = (y_prob.ravel() >= 0.5).astype(int)

    print("First 10 predicted labels:")
    print(y_pred[:10])

    print("\nFirst 10 actual labels:")
    print(y_test[:10])
else:
    print("Skipping label conversion: 'y_prob' and/or 'y_test' are not defined.")
    print("Please upload a text classification dataset, update the dataset path, and run all cells.")

Skipping label conversion: 'y_prob' and/or 'y_test' are not defined.
Please upload a text classification dataset, update the dataset path, and run all cells.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Safely evaluate metrics only if the required variables are defined
if 'y_test' in globals() and 'y_pred' in globals():
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test, y_pred, zero_division=0
    )

    recall = recall_score(
        y_test, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )

    print("MODEL EVALUATION RESULTS")
    print("------------------------")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
else:
    print("Skipping evaluation: 'y_test' and/or 'y_pred' are not defined.")
    print("To fix this, please upload a valid text classification dataset, update the dataset path in the beginning, and run all cells.")

Skipping evaluation: 'y_test' and/or 'y_pred' are not defined.
To fix this, please upload a valid text classification dataset, update the dataset path in the beginning, and run all cells.


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Safely calculate and plot confusion matrix only if variables exist
if 'y_test' in globals() and 'y_pred' in globals():
    # Calculate the confusion matrix
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    # Draw the confusion matrix
    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Ham", "Spam"],
        yticklabels=["Ham", "Spam"]
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.title("Simple RNN - SMS Spam Confusion Matrix")

    plt.show()

    print("Confusion matrix values:")
    print(cm)
else:
    print("Skipping confusion matrix plot: 'y_test' and/or 'y_pred' are not defined.")
    print("Please upload a text classification dataset (e.g. SMS Spam dataset), configure the filepath in cell SjcHLfZr0fw7, and run all cells.")

Skipping confusion matrix plot: 'y_test' and/or 'y_pred' are not defined.
Please upload a text classification dataset (e.g. SMS Spam dataset), configure the filepath in cell SjcHLfZr0fw7, and run all cells.


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Safely calculate and plot confusion matrix only if variables exist
if 'y_test' in globals() and 'y_pred' in globals():
    # Calculate the confusion matrix
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    # Draw the confusion matrix
    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Ham", "Spam"],
        yticklabels=["Ham", "Spam"]
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.title("Simple RNN - SMS Spam Confusion Matrix")

    plt.show()

    print("Confusion matrix values:")
    print(cm)
else:
    print("Skipping confusion matrix plot: 'y_test' and/or 'y_pred' are not defined.")
    print("Please upload a text classification dataset (e.g., SMS Spam dataset), configure the filepath in cell SjcHLfZr0fw7, and run all cells.")

Skipping confusion matrix plot: 'y_test' and/or 'y_pred' are not defined.
Please upload a text classification dataset (e.g., SMS Spam dataset), configure the filepath in cell SjcHLfZr0fw7, and run all cells.


In [ ]:
def predict_sms(message):

    # Convert the message to a numerical sequence
    sequence = vectorizer(
        np.array([message])
    ).numpy()

    # Predict the spam probability
    probability = model.predict(
        sequence,
        verbose=0
    )[0][0]

    # Convert the probability to a label
    if probability >= 0.5:
        label = "SPAM"
    else:
        label = "HAM"

    print("SMS message:", message)
    print("Predicted label:", label)
    print(f"Spam probability: {probability:.2%}")


# Try a new SMS message
predict_sms(
    "Congratulations! You have won a free prize. Claim now!"
)

FailedPreconditionError: Exception encountered when calling TextVectorization.call().

[1m{{function_node __wrapped__LookupTableFindV2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Table not initialized. [Op:LookupTableFindV2] name: [0m

Arguments received by TextVectorization.call():
  • inputs=array(['Congratulations! You have won a free prize. Claim now!'],
      dtype='<U54')